# ✈️ Caso Práctico 2 — AeroMetrics: Benchmark de formatos

## 🌍 Contexto de negocio

**AeroMetrics Analytics** ha recibido un encargo del Ministerio de Transporte: analizar los datos históricos de retrasos y cancelaciones de vuelos en EE.UU. entre **2009 y 2018**.

Los datos llegan en formato CSV, uno por año. El equipo debe:

1. **Ingestar** los CSV anuales en Spark.
2. **Convertirlos** a 4 formatos: CSV consolidado, Parquet, ORC y Avro.
3. **Medir** rendimiento de cada formato (escritura, lectura, tamaño en disco, consulta selectiva).
4. **Construir un Data Lake mínimo** con Parquet particionado por año.
5. **Redactar un informe comparativo**.

---

## 📥 Paso 0 — Descarga del dataset

**Fuente:** [Airline Delay and Cancellation Data 2009-2018 (Kaggle)](https://www.kaggle.com/datasets/yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018)

El dataset completo contiene **10 ficheros CSV** (uno por año) y ocupa ~6 GB. Para este caso práctico **basta con 3 años** (≈ 17 M filas, benchmark plenamente representativo).

**Pasos:**

1. Acceder a Kaggle con cuenta gratuita y descargar al menos `2016.csv`, `2017.csv` y `2018.csv`.
2. Crear la carpeta `csv_raw/` **dentro de esta misma carpeta del caso práctico** (ya está incluida en el repo, pero por si la borras):

```powershell
New-Item -ItemType Directory -Force -Path ".\caso-practico-2-aerometrics\csv_raw"
```

3. Mover los CSV descargados a `caso-practico-2-aerometrics\csv_raw\`.

**Estructura final esperada (autocontenida en la carpeta del caso):**

```text
caso-practico-2-aerometrics\
  caso-practico-2-aerometrics.ipynb
  README.md
  csv_raw\
    2016.csv
    2017.csv
    2018.csv
  salida\           # se generará al ejecutar el notebook

```> ⚠️ **Importante:** este notebook NO descarga datos. Necesitas tener los CSV en `caso-practico-2-aerometrics/csv_raw/` antes de la **Parte 5**. Las partes 1–4 sí se pueden ejecutar sin datos.


---

## 🔧 Parte 1 — Inicialización del entorno

In [1]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import $ivy.`org.apache.spark::spark-avro:4.1.1`

import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

import java.nio.file.{Files, Paths}
import java.io.File

val spark = SparkSession.builder()
  .appName("AeroMetrics_Benchmark")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "8")
  .config("spark.ui.showConsoleProgress", "false")
  // ↓ Mitiga SparkOutOfMemoryError en escrituras particionadas con poca heap (Almond local).
  .config("spark.sql.files.maxRecordsPerFile", "1000000")
  .config("spark.memory.fraction", "0.8")
  .getOrCreate()

import spark.implicits._

spark.sparkContext.setLogLevel("ERROR")

println(s"✅ Spark ${spark.version} listo — AeroMetrics Analytics")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/06 13:10:29 INFO SparkContext: Running Spark version 4.1.1
26/05/06 13:10:29 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/05/06 13:10:29 INFO SparkContext: Java version 17.0.18+8
26/05/06 13:10:30 INFO ResourceUtils: ==============================================================
26/05/06 13:10:30 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/06 13:10:30 INFO ResourceUtils: ==============================================================
26/05/06 13:10:30 INFO SparkContext: Submitted application: AeroMetrics_Benchmark
26/05/06 13:10:30 INFO SecurityManager: Changing view acls to: gre
26/05/06 13:10:30 INFO SecurityManager: Changing modify acls to: gre
26/05/06 13:10:30 INFO SecurityManager: Changing view acls groups to: gre
26/05/06 13:10:30 INFO SecurityManager: Changing modify acls groups to: gre
26/05/06 13:10:30 INFO SecurityManager: SecurityManager: authenticat

✅ Spark 4.1.1 listo — AeroMetrics Analytics


import $ivy.$
import $ivy.$
import $ivy.$
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import java.nio.file.{Files, Paths}
import java.io.File
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@3cabbf83
import spark.implicits._

---

## 🔧 Parte 2 — Funciones auxiliares de benchmark

In [2]:
// Mide el tiempo de ejecución de cualquier bloque de código
def medirTiempo[T](bloque: => T): (T, Long) = {
  val inicio = System.nanoTime()
  val resultado = bloque
  val fin = System.nanoTime()
  val tiempoMs = (fin - inicio) / 1000000L
  (resultado, tiempoMs)
}

// Calcula el tamaño total en bytes de una carpeta (suma todos los ficheros recursivamente)
def tamanoBytes(ruta: String): Long = {
  val p = Paths.get(ruta)
  if (!Files.exists(p)) 0L
  else {
    Files.walk(p)
      .filter(Files.isRegularFile(_))
      .mapToLong(Files.size(_))
      .sum()
  }
}

// Formatea bytes a una unidad legible
def formatoTamano(bytes: Long): String = {
  if      (bytes < 1024L)              f"$bytes B"
  else if (bytes < 1048576L)           f"${bytes / 1024.0}%.2f KB"
  else if (bytes < 1073741824L)        f"${bytes / 1048576.0}%.2f MB"
  else                                 f"${bytes / 1073741824.0}%.2f GB"
}

println("✅ Funciones de benchmark definidas: medirTiempo, tamanoBytes, formatoTamano")

✅ Funciones de benchmark definidas: medirTiempo, tamanoBytes, formatoTamano


defined function medirTiempo
defined function tamanoBytes
defined function formatoTamano

---

## 🔧 Parte 3 — Crear estructura de carpetas de salida

In [3]:
// Rutas relativas a la carpeta del notebook → caso autocontenido y portable.
val rutaBase    = "."
val rutaCSVRaw  = s"$rutaBase/csv_raw"
val rutaSalida  = s"$rutaBase/salida"

val carpetas = List(
  rutaCSVRaw,
  s"$rutaSalida/consolidado_csv",
  s"$rutaSalida/parquet_sin_partir",
  s"$rutaSalida/orc",
  s"$rutaSalida/avro",
  s"$rutaSalida/parquet_particionado"
)

carpetas.foreach { c =>
  Files.createDirectories(Paths.get(c))
  println(s"  ✅ $c")
}

println(s"\n📁 Estructura de salida lista en: $rutaSalida")

// Verificar si los CSV de Kaggle están presentes
val csvDescargados = new File(rutaCSVRaw).listFiles().filter(_.getName.endsWith(".csv"))
if (csvDescargados.isEmpty) {
  println(s"\n⚠️  No hay CSV en $rutaCSVRaw. Descarga el dataset de Kaggle antes de continuar con la Parte 5.")
} else {
  println(s"\n✅ Ficheros CSV detectados (${csvDescargados.length}):")
  csvDescargados.foreach(f => println(s"   - ${f.getName} (${formatoTamano(f.length())})"))
}

  ✅ ./csv_raw
  ✅ ./salida/consolidado_csv
  ✅ ./salida/parquet_sin_partir
  ✅ ./salida/orc
  ✅ ./salida/avro
  ✅ ./salida/parquet_particionado

📁 Estructura de salida lista en: ./salida

✅ Ficheros CSV detectados (3):
   - 2016.csv (662,42 MB)
   - 2017.csv (669,73 MB)
   - 2018.csv (851,62 MB)


rutaBase: String = "."
rutaCSVRaw: String = "./csv_raw"
rutaSalida: String = "./salida"
carpetas: List[String] = List(
  "./csv_raw",
  "./salida/consolidado_csv",
  "./salida/parquet_sin_partir",
  "./salida/orc",
  "./salida/avro",
  "./salida/parquet_particionado"
)
csvDescargados: Array[File] = Array(
  .\csv_raw\2016.csv,
  .\csv_raw\2017.csv,
  .\csv_raw\2018.csv
)

---

## 📂 Parte 4 — Schema manual de 30 campos

Definir el schema explícitamente evita que Spark tenga que leer el fichero dos veces para inferirlo. Con datasets de 600 MB por año la diferencia es **muy notable**.

In [4]:
val schemaVuelos = StructType(List(
  StructField("FL_DATE",             StringType,  nullable = true),
  StructField("OP_CARRIER",          StringType,  nullable = true),
  StructField("OP_CARRIER_FL_NUM",   IntegerType, nullable = true),
  StructField("ORIGIN",              StringType,  nullable = true),
  StructField("DEST",                StringType,  nullable = true),
  StructField("CRS_DEP_TIME",        IntegerType, nullable = true),
  StructField("DEP_TIME",            DoubleType,  nullable = true),
  StructField("DEP_DELAY",           DoubleType,  nullable = true),
  StructField("TAXI_OUT",            DoubleType,  nullable = true),
  StructField("WHEELS_OFF",          DoubleType,  nullable = true),
  StructField("WHEELS_ON",           DoubleType,  nullable = true),
  StructField("TAXI_IN",             DoubleType,  nullable = true),
  StructField("CRS_ARR_TIME",        IntegerType, nullable = true),
  StructField("ARR_TIME",            DoubleType,  nullable = true),
  StructField("ARR_DELAY",           DoubleType,  nullable = true),
  StructField("CANCELLED",           DoubleType,  nullable = true),
  StructField("CANCELLATION_CODE",   StringType,  nullable = true),
  StructField("DIVERTED",            DoubleType,  nullable = true),
  StructField("CRS_ELAPSED_TIME",    DoubleType,  nullable = true),
  StructField("ACTUAL_ELAPSED_TIME", DoubleType,  nullable = true),
  StructField("AIR_TIME",            DoubleType,  nullable = true),
  StructField("FLIGHTS",             DoubleType,  nullable = true),
  StructField("DISTANCE",            DoubleType,  nullable = true),
  StructField("DISTANCE_GROUP",      IntegerType, nullable = true),
  StructField("CARRIER_DELAY",       DoubleType,  nullable = true),
  StructField("WEATHER_DELAY",       DoubleType,  nullable = true),
  StructField("NAS_DELAY",           DoubleType,  nullable = true),
  StructField("SECURITY_DELAY",      DoubleType,  nullable = true),
  StructField("LATE_AIRCRAFT_DELAY", DoubleType,  nullable = true)
))

println(s"✅ Schema definido con ${schemaVuelos.fields.length} campos")

✅ Schema definido con 29 campos


schemaVuelos: StructType = Seq(
  StructField(
    name = "FL_DATE",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "OP_CARRIER",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "OP_CARRIER_FL_NUM",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "ORIGIN",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "DEST",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "CRS_DEP_TIME",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "DEP_TIME",
...

---

## 📂 Parte 5 — Leer todos los CSV y añadir columna `ANIO`

Spark lee la carpeta `csv_raw/*.csv` con una sola instrucción y une todos los años en un único DataFrame.

> ⚠️ Esta celda requiere los CSV descargados de Kaggle en `caso-practico-2-aerometrics/csv_raw/`.

In [5]:
val (dfRaw, tiempoLecturaCSV) = medirTiempo {
  spark.read
    .option("header", "true")
    .option("encoding", "UTF-8")
    .schema(schemaVuelos)
    .csv(s"$rutaCSVRaw/*.csv")
}

// Añadir columna ANIO extraída de FL_DATE (formato "YYYY-MM-DD")
val dfVuelos = dfRaw.withColumn("ANIO", substring(col("FL_DATE"), 1, 4).cast(IntegerType))

// Cachear porque vamos a reutilizarlo en muchas escrituras
dfVuelos.cache()
val totalFilas = dfVuelos.count()

println(s"=== Dataset cargado ===")
println(s"  Filas totales        : $totalFilas")
println(s"  Columnas             : ${dfVuelos.columns.length}")
println(s"  Tiempo lectura CSV   : $tiempoLecturaCSV ms")
println()
dfVuelos.printSchema()
dfVuelos.show(3, truncate = true)

=== Dataset cargado ===
  Filas totales        : 18505725
  Columnas             : 30
  Tiempo lectura CSV   : 3844 ms

root
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (null

dfRaw: org.apache.spark.sql.package.DataFrame = [FL_DATE: string, OP_CARRIER: string ... 27 more fields]
tiempoLecturaCSV: Long = 3844L
dfVuelos: org.apache.spark.sql.package.DataFrame = [FL_DATE: string, OP_CARRIER: string ... 28 more fields]
res5_2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [FL_DATE: string, OP_CARRIER: string ... 28 more fields]
totalFilas: Long = 18505725L

---

## ✍️ Parte 6 — Escritura en los cuatro formatos

### 6.1 — CSV consolidado

In [6]:
val rutaCSVOut = s"$rutaSalida/consolidado_csv"

val (_, tEscrituraCSV) = medirTiempo {
  dfVuelos.write
    .mode("overwrite")
    .option("header", "true")
    .csv(rutaCSVOut)
}

println(s"✅ CSV escrito en $tEscrituraCSV ms")
println(s"   Ruta: $rutaCSVOut")

✅ CSV escrito en 54026 ms
   Ruta: ./salida/consolidado_csv


rutaCSVOut: String = "./salida/consolidado_csv"
tEscrituraCSV: Long = 54026L

### 6.2 — Parquet (sin partición)

In [7]:
val rutaParquet = s"$rutaSalida/parquet_sin_partir"

val (_, tEscrituraParquet) = medirTiempo {
  dfVuelos.write
    .mode("overwrite")
    .parquet(rutaParquet)
}

println(s"✅ Parquet escrito en $tEscrituraParquet ms")
println(s"   Ruta: $rutaParquet")

✅ Parquet escrito en 38140 ms
   Ruta: ./salida/parquet_sin_partir


rutaParquet: String = "./salida/parquet_sin_partir"
tEscrituraParquet: Long = 38140L

### 6.3 — ORC

In [8]:
val rutaORC = s"$rutaSalida/orc"

val (_, tEscrituraORC) = medirTiempo {
  dfVuelos.write
    .mode("overwrite")
    .orc(rutaORC)
}

println(s"✅ ORC escrito en $tEscrituraORC ms")
println(s"   Ruta: $rutaORC")

✅ ORC escrito en 44580 ms
   Ruta: ./salida/orc


rutaORC: String = "./salida/orc"
tEscrituraORC: Long = 44580L

### 6.4 — Avro

In [9]:
val rutaAvro = s"$rutaSalida/avro"

val (_, tEscrituraAvro) = medirTiempo {
  dfVuelos.write
    .mode("overwrite")
    .format("avro")
    .save(rutaAvro)
}

println(s"✅ Avro escrito en $tEscrituraAvro ms")
println(s"   Ruta: $rutaAvro")

✅ Avro escrito en 40481 ms
   Ruta: ./salida/avro


rutaAvro: String = "./salida/avro"
tEscrituraAvro: Long = 40481L

---

## 🗂️ Parte 7 — Parquet particionado por `ANIO` (Data Lake)

El paso más relevante en producción: cualquier consulta que filtre por año leerá únicamente la subcarpeta correspondiente (**partition pruning**).

In [10]:
val rutaParquetParticionado = s"$rutaSalida/parquet_particionado"

// Liberamos el caché ANTES del shuffle para dejar memoria libre al particionado.
if (dfVuelos.storageLevel.useMemory) dfVuelos.unpersist()

// 🔑 repartition($"ANIO") agrupa todos los registros del mismo año en la misma
// partición Spark → cada task escribe a UNA sola subcarpeta, abriendo un único
// writer Parquet en lugar de uno por año. Esto evita el SparkOutOfMemoryError
// típico de las escrituras particionadas con poca heap (kernel Almond local).
val (_, tEscrituraParticionado) = medirTiempo {
  dfVuelos
    .repartition(col("ANIO"))
    .write
    .mode("overwrite")
    .partitionBy("ANIO")
    .parquet(rutaParquetParticionado)
}

println(s"✅ Parquet particionado escrito en $tEscrituraParticionado ms")
println(s"   Ruta: $rutaParquetParticionado\n")

// Volvemos a cachearlo si lo vamos a reutilizar en pasos posteriores.
dfVuelos.cache()
dfVuelos.count()

println("Estructura de particiones generada:")
println(s"  $rutaParquetParticionado/")
new File(rutaParquetParticionado)
  .listFiles()
  .filter(_.isDirectory)
  .map(_.getName)
  .sorted
  .foreach(c => println(s"    $c/"))

✅ Parquet particionado escrito en 493994 ms
   Ruta: ./salida/parquet_particionado

Estructura de particiones generada:
  ./salida/parquet_particionado/
    ANIO=2016/
    ANIO=2017/
    ANIO=2018/


rutaParquetParticionado: String = "./salida/parquet_particionado"
res10_1: Any = [FL_DATE: string, OP_CARRIER: string ... 28 more fields]
tEscrituraParticionado: Long = 493994L
res10_5: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [FL_DATE: string, OP_CARRIER: string ... 28 more fields]
res10_6: Long = 18505725L

> 💡 Spark añade automáticamente el prefijo `ANIO=` a cada carpeta. Al leer, reconstruye la columna `ANIO` desde el nombre de la subcarpeta sin que tengas que hacer nada.

---

## 📖 Parte 8 — Tiempos de lectura completa

Forzamos la materialización con `.count()` para medir lectura completa.

In [11]:
// 8.1 — Lectura completa CSV
val (filasCSV, tLecturaCSV) = medirTiempo {
  spark.read
    .option("header", "true")
    .schema(schemaVuelos)
    .csv(rutaCSVOut)
    .count()
}
println(s"✅ Lectura CSV completa: $tLecturaCSV ms  | Filas: $filasCSV")

✅ Lectura CSV completa: 5482 ms  | Filas: 18505725


filasCSV: Long = 18505725L
tLecturaCSV: Long = 5482L

In [12]:
// 8.2 — Lectura completa Parquet
val (filasParquet, tLecturaParquet) = medirTiempo {
  spark.read.parquet(rutaParquet).count()
}
println(s"✅ Lectura Parquet completa: $tLecturaParquet ms  | Filas: $filasParquet")

✅ Lectura Parquet completa: 2959 ms  | Filas: 18505725


filasParquet: Long = 18505725L
tLecturaParquet: Long = 2959L

In [13]:
// 8.3 — Lectura completa ORC
val (filasORC, tLecturaORC) = medirTiempo {
  spark.read.orc(rutaORC).count()
}
println(s"✅ Lectura ORC completa: $tLecturaORC ms  | Filas: $filasORC")

✅ Lectura ORC completa: 2544 ms  | Filas: 18505725


filasORC: Long = 18505725L
tLecturaORC: Long = 2544L

In [14]:
// 8.4 — Lectura completa Avro
val (filasAvro, tLecturaAvro) = medirTiempo {
  spark.read.format("avro").load(rutaAvro).count()
}
println(s"✅ Lectura Avro completa: $tLecturaAvro ms  | Filas: $filasAvro")

✅ Lectura Avro completa: 10944 ms  | Filas: 18505725


filasAvro: Long = 18505725L
tLecturaAvro: Long = 10944L

---

## 🔍 Parte 9 — Consulta selectiva de pocas columnas

Consulta de referencia: **"Retraso medio de llegada y número de vuelos por aerolínea"**. Solo necesita 3 columnas (`OP_CARRIER`, `ARR_DELAY`, `CANCELLED`). Aquí los formatos columnares brillan.

In [15]:
// 9.1 — Consulta selectiva sobre CSV
val (resultCSV, tConsultaCSV) = medirTiempo {
  spark.read
    .option("header", "true")
    .schema(schemaVuelos)
    .csv(rutaCSVOut)
    .filter(col("CANCELLED") === 0.0)
    .groupBy("OP_CARRIER")
    .agg(
      count("*").as("total_vuelos"),
      round(avg("ARR_DELAY"), 2).as("retraso_medio_min")
    )
    .orderBy(desc("total_vuelos"))
    .collect()
}
println(s"✅ Consulta CSV: $tConsultaCSV ms — ${resultCSV.length} aerolíneas")

✅ Consulta CSV: 27309 ms — 18 aerolíneas


resultCSV: Array[org.apache.spark.sql.Row] = Array(
  [WN,3929253,4.49],
  [DL,2778985,-0.31],
  [AA,2689711,4.8],
  [OO,2057083,6.25],
  [UA,1734781,3.17],
  [EV,1005896,6.95],
  [B6,867633,10.19],
  [AS,603579,-0.65],
  [NK,461688,6.53],
  [F9,313710,9.51],
  [YX,305990,3.08],
  [MQ,285346,5.36],
  [OH,266587,8.24],
  [HA,240086,0.91],
  [9E,239562,4.45],
  [YV,209608,8.85],
  [VX,155637,8.44],
  [G4,95452,9.98]
)
tConsultaCSV: Long = 27309L

In [16]:
// 9.2 — Consulta selectiva sobre Parquet
val (resultParquet, tConsultaParquet) = medirTiempo {
  spark.read
    .parquet(rutaParquet)
    .filter(col("CANCELLED") === 0.0)
    .groupBy("OP_CARRIER")
    .agg(
      count("*").as("total_vuelos"),
      round(avg("ARR_DELAY"), 2).as("retraso_medio_min")
    )
    .orderBy(desc("total_vuelos"))
    .collect()
}
println(s"✅ Consulta Parquet: $tConsultaParquet ms — ${resultParquet.length} aerolíneas")

✅ Consulta Parquet: 3977 ms — 18 aerolíneas


resultParquet: Array[org.apache.spark.sql.Row] = Array(
  [WN,3929253,4.49],
  [DL,2778985,-0.31],
  [AA,2689711,4.8],
  [OO,2057083,6.25],
  [UA,1734781,3.17],
  [EV,1005896,6.95],
  [B6,867633,10.19],
  [AS,603579,-0.65],
  [NK,461688,6.53],
  [F9,313710,9.51],
  [YX,305990,3.08],
  [MQ,285346,5.36],
  [OH,266587,8.24],
  [HA,240086,0.91],
  [9E,239562,4.45],
  [YV,209608,8.85],
  [VX,155637,8.44],
  [G4,95452,9.98]
)
tConsultaParquet: Long = 3977L

In [17]:
// 9.3 — Consulta selectiva sobre ORC
val (resultORC, tConsultaORC) = medirTiempo {
  spark.read
    .orc(rutaORC)
    .filter(col("CANCELLED") === 0.0)
    .groupBy("OP_CARRIER")
    .agg(
      count("*").as("total_vuelos"),
      round(avg("ARR_DELAY"), 2).as("retraso_medio_min")
    )
    .orderBy(desc("total_vuelos"))
    .collect()
}
println(s"✅ Consulta ORC: $tConsultaORC ms — ${resultORC.length} aerolíneas")

✅ Consulta ORC: 5079 ms — 18 aerolíneas


resultORC: Array[org.apache.spark.sql.Row] = Array(
  [WN,3929253,4.49],
  [DL,2778985,-0.31],
  [AA,2689711,4.8],
  [OO,2057083,6.25],
  [UA,1734781,3.17],
  [EV,1005896,6.95],
  [B6,867633,10.19],
  [AS,603579,-0.65],
  [NK,461688,6.53],
  [F9,313710,9.51],
  [YX,305990,3.08],
  [MQ,285346,5.36],
  [OH,266587,8.24],
  [HA,240086,0.91],
  [9E,239562,4.45],
  [YV,209608,8.85],
  [VX,155637,8.44],
  [G4,95452,9.98]
)
tConsultaORC: Long = 5079L

In [18]:
// 9.4 — Consulta selectiva sobre Avro
val (resultAvro, tConsultaAvro) = medirTiempo {
  spark.read
    .format("avro")
    .load(rutaAvro)
    .filter(col("CANCELLED") === 0.0)
    .groupBy("OP_CARRIER")
    .agg(
      count("*").as("total_vuelos"),
      round(avg("ARR_DELAY"), 2).as("retraso_medio_min")
    )
    .orderBy(desc("total_vuelos"))
    .collect()
}
println(s"✅ Consulta Avro: $tConsultaAvro ms — ${resultAvro.length} aerolíneas")

✅ Consulta Avro: 12964 ms — 18 aerolíneas


resultAvro: Array[org.apache.spark.sql.Row] = Array(
  [WN,3929253,4.49],
  [DL,2778985,-0.31],
  [AA,2689711,4.8],
  [OO,2057083,6.25],
  [UA,1734781,3.17],
  [EV,1005896,6.95],
  [B6,867633,10.19],
  [AS,603579,-0.65],
  [NK,461688,6.53],
  [F9,313710,9.51],
  [YX,305990,3.08],
  [MQ,285346,5.36],
  [OH,266587,8.24],
  [HA,240086,0.91],
  [9E,239562,4.45],
  [YV,209608,8.85],
  [VX,155637,8.44],
  [G4,95452,9.98]
)
tConsultaAvro: Long = 12964L

In [19]:
// 9.5 — Consulta sobre Parquet PARTICIONADO con filtro por año (partition pruning)
val anioConsulta = 2018  // ← cambia el año según los datos descargados

val (resultParticionado, tConsultaParticionado) = medirTiempo {
  spark.read
    .parquet(rutaParquetParticionado)
    .filter(col("ANIO") === anioConsulta && col("CANCELLED") === 0.0)
    .groupBy("OP_CARRIER")
    .agg(
      count("*").as("total_vuelos"),
      round(avg("ARR_DELAY"), 2).as("retraso_medio_min")
    )
    .orderBy(desc("total_vuelos"))
    .collect()
}
println(s"✅ Consulta Parquet particionado (ANIO=$anioConsulta): $tConsultaParticionado ms")
println(s"   Aerolíneas encontradas: ${resultParticionado.length}")

✅ Consulta Parquet particionado (ANIO=2018): 2198 ms
   Aerolíneas encontradas: 18


anioConsulta: Int = 2018
resultParticionado: Array[org.apache.spark.sql.Row] = Array(
  [WN,1334277,4.52],
  [DL,945755,-0.29],
  [AA,901873,5.43],
  [OO,763527,7.04],
  [UA,616662,5.76],
  [YX,305990,3.08],
  [B6,298591,11.43],
  [MQ,285346,5.36],
  [OH,266587,8.24],
  [AS,243554,-0.5],
  [9E,239562,4.45],
  [YV,209608,8.85],
  [EV,197220,8.8],
  [NK,174441,5.17],
  [F9,117707,14.21],
  [G4,95452,9.98],
  [HA,83473,0.85],
  [VX,17237,1.73]
)
tConsultaParticionado: Long = 2198L

> 💡 Compara este tiempo con la consulta sobre Parquet sin particionar (9.2). La diferencia muestra el efecto del **partition pruning**: Spark descarta físicamente los datos de otros años sin leerlos.

---

## 📏 Parte 10 — Tamaños en disco

In [20]:
val tamCSVRaw       = tamanoBytes(rutaCSVRaw)
val tamCSVOut       = tamanoBytes(rutaCSVOut)
val tamParquet      = tamanoBytes(rutaParquet)
val tamORC          = tamanoBytes(rutaORC)
val tamAvro         = tamanoBytes(rutaAvro)
val tamParticionado = tamanoBytes(rutaParquetParticionado)

val sep = "=" * 56
println(sep)
println("   COMPARATIVA DE TAMAÑO EN DISCO")
println(sep)
println(f"  CSV original (raw)          : ${formatoTamano(tamCSVRaw)}%12s")
println(f"  CSV consolidado (Spark out) : ${formatoTamano(tamCSVOut)}%12s")
println(f"  Parquet (sin partición)     : ${formatoTamano(tamParquet)}%12s")
println(f"  ORC                         : ${formatoTamano(tamORC)}%12s")
println(f"  Avro                        : ${formatoTamano(tamAvro)}%12s")
println(f"  Parquet particionado        : ${formatoTamano(tamParticionado)}%12s")
println(sep)

if (tamCSVRaw > 0) {
  println("\nRatios de compresión respecto al CSV raw:")
  println(f"  Parquet  : ${tamCSVRaw.toDouble / tamParquet}%5.1fx menos espacio que CSV")
  println(f"  ORC      : ${tamCSVRaw.toDouble / tamORC}%5.1fx menos espacio que CSV")
  println(f"  Avro     : ${tamCSVRaw.toDouble / tamAvro}%5.1fx menos espacio que CSV")
}

   COMPARATIVA DE TAMAÑO EN DISCO
  CSV original (raw)          :      2,13 GB
  CSV consolidado (Spark out) :      2,24 GB
  Parquet (sin partición)     :    405,38 MB
  ORC                         :    421,71 MB
  Avro                        :      1,01 GB
  Parquet particionado        :    410,25 MB

Ratios de compresión respecto al CSV raw:
  Parquet  :   5,4x menos espacio que CSV
  ORC      :   5,2x menos espacio que CSV
  Avro     :   2,1x menos espacio que CSV


tamCSVRaw: Long = 2289841990L
tamCSVOut: Long = 2409352019L
tamParquet: Long = 425075155L
tamORC: Long = 442193598L
tamAvro: Long = 1080651995L
tamParticionado: Long = 430176598L
sep: String = "========================================================"

---

## 📊 Parte 11 — Tabla comparativa final

In [22]:
// ⚠️ La case class va en una celda separada de la que hace `.toDF()`.
// Si se define en la misma celda que la conversión, Almond produce
// `NullPointerException: Cannot invoke "Object.getClass()" because "outer" is null`
// porque el encoder de Spark no puede resolver el outer pointer del wrapper REPL.
case class ResultadoFormato(
  formato:      String,
  tEscrituraMs: Long,
  tLecturaMs:   Long,
  tConsultaMs:  Long,
  tamanoDisco:  String
)

defined class ResultadoFormato

In [23]:
val resultados = Seq(
  ResultadoFormato("CSV",                  tEscrituraCSV,          tLecturaCSV,     tConsultaCSV,          formatoTamano(tamCSVOut)),
  ResultadoFormato("Parquet (sin partir)", tEscrituraParquet,      tLecturaParquet, tConsultaParquet,      formatoTamano(tamParquet)),
  ResultadoFormato("ORC",                  tEscrituraORC,          tLecturaORC,     tConsultaORC,          formatoTamano(tamORC)),
  ResultadoFormato("Avro",                 tEscrituraAvro,         tLecturaAvro,    tConsultaAvro,         formatoTamano(tamAvro)),
  ResultadoFormato("Parquet particionado", tEscrituraParticionado, 0L,              tConsultaParticionado, formatoTamano(tamParticionado))
)

val sep2 = "=" * 90
println(sep2)
println(f"  ${"FORMATO"}%-25s ${"T.ESCRITURA"}%14s ${"T.LECTURA"}%12s ${"T.CONSULTA"}%13s ${"DISCO"}%12s")
println(sep2)
resultados.foreach { r =>
  val lectura = if (r.tLecturaMs == 0L) "  (n/a)" else f"${r.tLecturaMs}%8d ms"
  println(f"  ${r.formato}%-25s ${r.tEscrituraMs}%10d ms  $lectura%12s  ${r.tConsultaMs}%9d ms  ${r.tamanoDisco}%12s")
}
println(sep2)

// También como DataFrame para inspección visual.
// `toDF()` funciona ahora porque la case class está en una celda distinta.
println("\n=== Misma tabla como DataFrame ===")
resultados.toDF().show(false)

  FORMATO                      T.ESCRITURA    T.LECTURA    T.CONSULTA        DISCO
  CSV                            54026 ms       5482 ms      27309 ms       2,24 GB
  Parquet (sin partir)           38140 ms       2959 ms       3977 ms     405,38 MB
  ORC                            44580 ms       2544 ms       5079 ms     421,71 MB
  Avro                           40481 ms      10944 ms      12964 ms       1,01 GB
  Parquet particionado          493994 ms         (n/a)       2198 ms     410,25 MB

=== Misma tabla como DataFrame ===
+--------------------+------------+----------+-----------+-----------+
|formato             |tEscrituraMs|tLecturaMs|tConsultaMs|tamanoDisco|
+--------------------+------------+----------+-----------+-----------+
|CSV                 |54026       |5482      |27309      |2,24 GB    |
|Parquet (sin partir)|38140       |2959      |3977       |405,38 MB  |
|ORC                 |44580       |2544      |5079       |421,71 MB  |
|Avro                |40481       |

resultados: Seq[ResultadoFormato] = List(
  ResultadoFormato(
    formato = "CSV",
    tEscrituraMs = 54026L,
    tLecturaMs = 5482L,
    tConsultaMs = 27309L,
    tamanoDisco = "2,24 GB"
  ),
  ResultadoFormato(
    formato = "Parquet (sin partir)",
    tEscrituraMs = 38140L,
    tLecturaMs = 2959L,
    tConsultaMs = 3977L,
    tamanoDisco = "405,38 MB"
  ),
  ResultadoFormato(
    formato = "ORC",
    tEscrituraMs = 44580L,
    tLecturaMs = 2544L,
    tConsultaMs = 5079L,
    tamanoDisco = "421,71 MB"
  ),
  ResultadoFormato(
    formato = "Avro",
    tEscrituraMs = 40481L,
    tLecturaMs = 10944L,
    tConsultaMs = 12964L,
    tamanoDisco = "1,01 GB"
  ),
  ResultadoFormato(
    formato = "Parquet particionado",
    tEscrituraMs = 493994L,
    tLecturaMs = 0L,
    tConsultaMs = 2198L,
    tamanoDisco = "410,25 MB"
  )
)
sep2: String = "=========================================================================================="

**Ejemplo orientativo (valores con ~17 M filas):**

```text
  FORMATO                     T.ESCRITURA    T.LECTURA   T.CONSULTA      DISCO
  CSV                          45 230 ms     38 900 ms    52 100 ms     1.82 GB
  Parquet (sin partir)         28 100 ms      8 200 ms     6 400 ms   320.00 MB
  ORC                          31 500 ms      7 800 ms     5 900 ms   290.00 MB
  Avro                         35 200 ms     12 600 ms    18 300 ms   540.00 MB
  Parquet particionado         29 800 ms        (n/a)      1 200 ms   318.00 MB
```

> ⚠️ Tus tiempos variarán según hardware y nº de años. Lo importante es la **proporción relativa** entre formatos.

---

## 🔎 Parte 12 — Verificación del particionado con Spark SQL

In [24]:
spark.read
  .parquet(rutaParquetParticionado)
  .createOrReplaceTempView("vuelos_lake")

println("=== Consulta 1: total de vuelos por año (verificar particiones) ===")
spark.sql("""
  SELECT ANIO, COUNT(*) AS total_vuelos
  FROM vuelos_lake
  GROUP BY ANIO
  ORDER BY ANIO
""").show()

println("=== Consulta 2: top 10 rutas con más retraso medio en 2018 ===")
spark.sql("""
  SELECT
    ORIGIN,
    DEST,
    COUNT(*)                 AS total_vuelos,
    ROUND(AVG(ARR_DELAY), 1) AS retraso_medio_min,
    ROUND(SUM(CANCELLED), 0) AS vuelos_cancelados
  FROM vuelos_lake
  WHERE ANIO = 2018
    AND ARR_DELAY > 0
  GROUP BY ORIGIN, DEST
  ORDER BY retraso_medio_min DESC
  LIMIT 10
""").show(truncate = false)

=== Consulta 1: total de vuelos por año (verificar particiones) ===
+----+------------+
|ANIO|total_vuelos|
+----+------------+
|2016|     5617658|
|2017|     5674621|
|2018|     7213446|
+----+------------+

=== Consulta 2: top 10 rutas con más retraso medio en 2018 ===
+------+----+------------+-----------------+-----------------+
|ORIGIN|DEST|total_vuelos|retraso_medio_min|vuelos_cancelados|
+------+----+------------+-----------------+-----------------+
|RDM   |MFR |1           |1347.0           |0.0              |
|MDT   |HPN |1           |798.0            |0.0              |
|GRK   |ATL |6           |243.8            |0.0              |
|ISP   |MSP |7           |224.0            |0.0              |
|ICT   |DAY |1           |210.0            |0.0              |
|JFK   |JAC |2           |199.0            |0.0              |
|DSM   |PIA |1           |168.0            |0.0              |
|TVC   |EWR |52          |165.5            |0.0              |
|EGE   |IAH |22          |165.3    

---

## 📝 Parte 13 — Preguntas de reflexión

### 🅰️ Sobre los formatos

**1. ¿Qué formato tardó más en escribirse? ¿A qué se debe?**

Habitualmente **CSV** es el más lento de escribir, porque Spark debe **serializar cada valor a texto** y escapar separadores/saltos de línea fila a fila. ORC y Parquet, aunque comprimen y construyen estadísticas por bloque, escriben en binario por columnas, lo que en datasets grandes resulta más rápido. Avro queda en medio: escribe binario pero **fila a fila**, sin la ventaja columnar.

**2. ¿Qué formato ocupó menos espacio? ¿Por qué los columnares comprimen mejor que CSV?**

**Parquet y ORC** ocupan claramente menos (≈ 5-7× menos que CSV). Tres razones:
- **Codificación columnar**: valores del mismo tipo se almacenan juntos, los algoritmos de compresión (Snappy, Zstd, Zlib) explotan la redundancia mucho mejor.
- **Encoding inteligente** por columna: dictionary encoding para texto repetido, run-length para series, delta encoding para enteros consecutivos.
- **Sin overhead textual**: no se repiten cabeceras ni separadores en cada fila.

**3. ¿Qué formato fue más rápido para la consulta de 3 columnas? ¿Por qué?**

**Parquet y ORC** ganan por amplio margen. Almacenan los datos **por columnas**, así Spark lee únicamente los bytes de `OP_CARRIER`, `ARR_DELAY` y `CANCELLED` y **descarta físicamente las otras 27 columnas sin tocarlas**. CSV y Avro son por filas: hay que leer y deserializar cada fila completa para luego descartar 27 campos.

**4. Avro es binario y tiene schema embebido, pero fue más lento que Parquet/ORC en la consulta selectiva. ¿Por qué?**

Porque **Avro es binario por filas**, no columnar. Aunque la deserialización es eficiente, Spark sigue obligado a leer cada fila entera y desempaquetar **los 30 campos** aunque la consulta solo necesite 3. Avro está optimizado para casos donde se procesan **registros completos** (Kafka, eventos, intercambio entre sistemas), no para consultas analíticas.

### 🅱️ Sobre el particionado

**1. ¿Qué diferencia observaste entre Parquet sin particionar y Parquet particionado con filtro `ANIO=2018`?**

El particionado es **mucho más rápido** (a menudo ×3-5 con 3 años, ×10 o más con los 10 años). Spark hace **partition pruning**: ignora completamente las carpetas `ANIO=2016/` y `ANIO=2017/` y solo escanea `ANIO=2018/`. La cantidad de datos físicamente leídos del disco se divide por el número de años no consultados.

**2. ¿Qué ocurriría si particionaras por `OP_CARRIER` (aerolínea)?**

Habría una carpeta por cada aerolínea (decenas). Las consultas que filtran por `OP_CARRIER` serían rapidísimas, **pero**:
- Las que filtran por año o por ruta perderían el beneficio.
- Algunas aerolíneas tienen muchísimos vuelos (Delta, Southwest) y otras muy pocos → particiones desbalanceadas → **skew** en los jobs de Spark.
- Un solo job podría leer todas las particiones igualmente, perdiendo la ventaja.

**3. ¿Por qué no se debe particionar por una columna con demasiados valores únicos?**

Porque cada valor único genera **una carpeta y al menos un fichero** (lo que se llama el problema de los *small files*). Particionar por `id_vuelo`, `FL_DATE` o cualquier campo con millones de valores únicos:
- Crea millones de ficheros pequeños → sobrecarga el NameNode (en HDFS) o el listado de objetos (en S3).
- Cada fichero tiene metadatos propios; el coste fijo se vuelve dominante.
- Spark tarda más en **planificar** la lectura que en leerla.
- **Regla práctica:** particionar por columnas con cardinalidad **baja-media** (años, países, categorías), no por IDs únicos.

### 🅲 Sobre el pipeline

**1. ¿Por qué definir el schema manualmente en lugar de `inferSchema = true` con ficheros de 600 MB?**

Con `inferSchema = true` Spark hace **dos pasadas** sobre el fichero: una para deducir tipos y otra para cargarlo. Con 6 GB esto duplica el tiempo de lectura inicial. Además:
- La inferencia puede equivocarse (típico problema con `FL_DATE` quedando como `String` o números con decimales mal interpretados).
- Si llegan ficheros nuevos con valores diferentes, los tipos pueden cambiar entre ejecuciones, rompiendo el pipeline silenciosamente.
- El schema explícito es **autodocumentación** del contrato del dato.

**2. ¿Qué modo de escritura usarías para añadir 2019 al Parquet particionado sin borrar los demás?**

**`append`**, idealmente combinado con `partitionBy("ANIO")`:

```scala
df2019.write
  .mode("append")
  .partitionBy("ANIO")
  .parquet(rutaParquetParticionado)
```

Spark añadirá una nueva subcarpeta `ANIO=2019/` sin tocar las existentes.

Para producción real conviene usar `dynamicPartitionOverwrite` o **Delta Lake / Iceberg** para reescribir solo las particiones afectadas de forma transaccional. Con Spark vainilla, el modo `append` es suficiente si sabemos que los datos nuevos no se solapan con años existentes.

---

## 🛑 Cierre

In [25]:
spark.stop()
println("✅ Caso Práctico 2 — AeroMetrics completado")

✅ Caso Práctico 2 — AeroMetrics completado
